# Plus d'historique par station + features calendaires — PFE

Suite à `04_train_multi_horizon.ipynb` (météo, non concluante avec le volume
actuel de données), on teste deux nouvelles pistes moins coûteuses et plus
robustes :

1. **Plus de lags** (historique récent par station) — capte mieux la
   dynamique locale d'une station juste avant l'instant de prédiction.
2. **Features calendaires** (weekend, jour férié) — pas besoin d'API externe,
   toujours disponibles, et souvent plus déterminantes que la météo pour
   expliquer les habitudes de déplacement.

On commence par un diagnostic honnête : avec combien de jours et de points
par station travaille-t-on réellement ? Ça conditionne si ces features ont
une chance d'être utiles ou non.


In [1]:
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()
DB_URL = (
    f"postgresql+psycopg2://{os.environ['DB_USER']}:{os.environ['DB_PASSWORD']}"
    f"@{os.environ['DB_HOST']}:{os.environ['DB_PORT']}/{os.environ['DB_NAME']}"
)
engine = create_engine(DB_URL)

status = pd.read_sql("""
    select station_id, collected_at, num_bikes_available, num_docks_available
    from dbt_dev.silver_station_status
    order by station_id, collected_at
""", engine, parse_dates=["collected_at"])

print(f"Status : {len(status):,} lignes, {status['station_id'].nunique()} stations")


Status : 127,266 lignes, 1516 stations


## 1. Diagnostic : combien d'historique a-t-on vraiment ?

In [2]:
date_min = status["collected_at"].min()
date_max = status["collected_at"].max()
n_days = (date_max - date_min).total_seconds() / 86400
rows_per_station = status.groupby("station_id").size()

print(f"Période couverte : {date_min} -> {date_max}  ({n_days:.2f} jours)")
print(f"Jours distincts observés : {status['collected_at'].dt.date.nunique()}")
print(f"Jours de la semaine distincts observés : {sorted(status['collected_at'].dt.dayofweek.unique())} "
      "(0=lundi ... 6=dimanche)")
print(f"Points par station : min={rows_per_station.min()}, "
      f"médiane={rows_per_station.median():.0f}, max={rows_per_station.max()}")


Période couverte : 2026-09-21 23:06:06.633491+00:00 -> 2026-09-23 10:41:14.636291+00:00  (1.48 jours)
Jours distincts observés : 3
Jours de la semaine distincts observés : [np.int32(0), np.int32(1), np.int32(2)] (0=lundi ... 6=dimanche)
Points par station : min=6, médiane=84, max=84


**Lecture honnête du diagnostic ci-dessus** (à interpréter selon le résultat) :
- Si peu de jours distincts sont couverts, un modèle ne peut pas vraiment
  apprendre un effet "weekend" ou "jour férié" — il n'aura simplement pas vu
  assez d'exemples de chaque cas. On ajoute quand même les features (le code
  est prêt pour quand plus de données seront collectées), mais on s'attend
  à ce qu'elles n'apportent rien de mesurable pour l'instant, exactement
  comme pour la météo au notebook 04.
- Le nombre de points par station donne la limite haute raisonnable pour le
  nombre de lags : inutile de demander 20 lags si une station n'a que
  30 points d'historique.


## 2. Features calendaires (sans dépendance externe)

- `is_weekend` : samedi/dimanche
- `is_holiday` : jours fériés français (liste figée, pas besoin d'API)


In [3]:
# Jours fériés français (métropole) — liste figée, aucune dépendance externe.
# Couvre large (2025-2027) pour rester valable pendant toute la durée du projet.
JOURS_FERIES = pd.to_datetime([
    "2025-01-01", "2025-04-21", "2025-05-01", "2025-05-08", "2025-05-29",
    "2025-06-09", "2025-07-14", "2025-08-15", "2025-11-01", "2025-11-11", "2025-12-25",
    "2026-01-01", "2026-04-06", "2026-05-01", "2026-05-08", "2026-05-14",
    "2026-05-25", "2026-07-14", "2026-08-15", "2026-11-01", "2026-11-11", "2026-12-25",
    "2027-01-01", "2027-03-29", "2027-05-01", "2027-05-08", "2027-05-06",
    "2027-05-17", "2027-07-14", "2027-08-15", "2027-11-01", "2027-11-11", "2027-12-25",
]).date

def add_calendar_features(df):
    df = df.copy()
    df["is_weekend"] = (df["collected_at"].dt.dayofweek >= 5).astype(int)
    df["is_holiday"] = df["collected_at"].dt.date.isin(JOURS_FERIES).astype(int)
    return df

status = add_calendar_features(status)
print(status[["collected_at", "is_weekend", "is_holiday"]].drop_duplicates().shape[0], "combinaisons distinctes")
print("Répartition is_weekend :", status["is_weekend"].value_counts().to_dict())
print("Répartition is_holiday :", status["is_holiday"].value_counts().to_dict())


84 combinaisons distinctes
Répartition is_weekend : {0: 127266}
Répartition is_holiday : {0: 127266}


## 3. Plus de lags par station

On passe de 3 à 6 lags (si l'historique le permet d'après le diagnostic
ci-dessus), pour capter une dynamique un peu plus longue juste avant
l'instant de prédiction.

In [4]:
N_LAGS = 6 if rows_per_station.min() >= 20 else 3
print(f"N_LAGS retenu : {N_LAGS} (min de points/station observé : {rows_per_station.min()})")

status_sorted = status.sort_values(["station_id", "collected_at"]).reset_index(drop=True)
status_sorted["hour"] = status_sorted["collected_at"].dt.hour
status_sorted["day_of_week"] = status_sorted["collected_at"].dt.dayofweek

for lag in range(1, N_LAGS + 1):
    status_sorted[f"lag_{lag}"] = status_sorted.groupby("station_id")["num_bikes_available"].shift(lag)

FEATURES = (
    ["hour", "day_of_week", "is_weekend", "is_holiday"]
    + [f"lag_{lag}" for lag in range(1, N_LAGS + 1)]
)
print("Features :", FEATURES)


N_LAGS retenu : 3 (min de points/station observé : 6)
Features : ['hour', 'day_of_week', 'is_weekend', 'is_holiday', 'lag_1', 'lag_2', 'lag_3']


## 4. Entraînement par horizon, comparaison honnête à la baseline

Même méthodologie que le notebook 04 (TimeSeriesSplit, comparaison à la
persistance), avec ce nouveau jeu de features (plus de lags + calendrier,
sans météo cette fois pour isoler l'effet de ces deux nouvelles pistes).

In [5]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

diffs = (
    status_sorted.groupby("station_id")["collected_at"]
    .diff().dt.total_seconds() / 60
)
STEP_MINUTES = float(diffs.median())
HORIZONS_MIN = [15, 30, 60]
STEPS_BY_HORIZON = {h: max(1, round(h / STEP_MINUTES)) for h in HORIZONS_MIN}
print(f"Intervalle médian observé : {STEP_MINUTES:.1f} min -> pas par horizon : {STEPS_BY_HORIZON}")

N_SPLITS = 3
results_by_horizon = {}

for horizon_min, steps in STEPS_BY_HORIZON.items():
    d = status_sorted.copy()
    d["target"] = d.groupby("station_id")["num_bikes_available"].shift(-steps)
    dataset = d.dropna(subset=[f"lag_{l}" for l in range(1, N_LAGS + 1)] + ["target"])
    dataset = dataset.sort_values("collected_at").reset_index(drop=True)

    X = dataset[FEATURES]
    y = dataset["target"]
    baseline_pred = dataset["lag_1"]

    tscv = TimeSeriesSplit(n_splits=N_SPLITS)
    model_maes, baseline_maes = [], []

    for train_idx, test_idx in tscv.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        model = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42)
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        model_maes.append(mean_absolute_error(y_test, pred))
        baseline_maes.append(mean_absolute_error(y_test, baseline_pred.iloc[test_idx]))

    mae_model = float(np.mean(model_maes))
    mae_baseline = float(np.mean(baseline_maes))
    use_model = mae_model < mae_baseline

    results_by_horizon[horizon_min] = {
        "steps": steps,
        "mae_model": round(mae_model, 3),
        "mae_baseline": round(mae_baseline, 3),
        "use_model": use_model,
        "n_train_rows": int(len(dataset)),
        "n_lags": N_LAGS,
    }
    verdict = "modèle retenu" if use_model else "baseline retenue"
    print(f"Horizon {horizon_min:>2} min ({steps} pas) : "
          f"MAE modèle={mae_model:.3f} vs MAE baseline={mae_baseline:.3f} -> {verdict}")


Intervalle médian observé : 5.0 min -> pas par horizon : {15: 3, 30: 6, 60: 12}
Horizon 15 min (3 pas) : MAE modèle=3.901 vs MAE baseline=1.930 -> baseline retenue
Horizon 30 min (6 pas) : MAE modèle=3.733 vs MAE baseline=2.883 -> baseline retenue
Horizon 60 min (12 pas) : MAE modèle=4.996 vs MAE baseline=4.526 -> baseline retenue


## 5. Comparaison avec le notebook 04 (météo) et sauvegarde

On ne sauvegarde que les modèles qui battent réellement leur baseline, comme
pour les notebooks précédents — même logique de repli honnête sur la
persistance si ce n'est pas le cas.

In [6]:
summary = pd.DataFrame(results_by_horizon).T
summary.index.name = "horizon_min"
print(summary)

import joblib, json as jsonlib
os.makedirs("../ml_models", exist_ok=True)

for horizon_min, steps in STEPS_BY_HORIZON.items():
    info = results_by_horizon[horizon_min]
    if not info["use_model"]:
        continue
    d = status_sorted.copy()
    d["target"] = d.groupby("station_id")["num_bikes_available"].shift(-steps)
    dataset = d.dropna(subset=[f"lag_{l}" for l in range(1, N_LAGS + 1)] + ["target"])
    final_model = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42)
    final_model.fit(dataset[FEATURES], dataset["target"])
    path = f"../ml_models/xgb_h{horizon_min}_v2.joblib"
    joblib.dump(final_model, path)
    print(f"Horizon {horizon_min} min -> sauvegardé dans {path}")

metrics_v2_path = "../ml_models/metrics_v2_calendar_lags.json"
with open(metrics_v2_path, "w", encoding="utf-8") as f:
    jsonlib.dump({
        "step_minutes_observed": STEP_MINUTES,
        "features": FEATURES,
        "n_lags": N_LAGS,
        "horizons": results_by_horizon,
    }, f, ensure_ascii=False, indent=2)
print(f"Métriques sauvegardées dans {metrics_v2_path}")
print()
print("Comparaison rapide avec le notebook 04 (météo, N_LAGS=3) :")
print("  -> voir ml_models/metrics.json pour les chiffres météo")
print("  -> voir ml_models/metrics_v2_calendar_lags.json pour ces nouveaux chiffres")


            steps mae_model mae_baseline use_model n_train_rows n_lags
horizon_min                                                           
15              3     3.901         1.93     False       118170      3
30              6     3.733        2.883     False       113625      3
60             12     4.996        4.526     False       104535      3
Métriques sauvegardées dans ../ml_models/metrics_v2_calendar_lags.json

Comparaison rapide avec le notebook 04 (météo, N_LAGS=3) :
  -> voir ml_models/metrics.json pour les chiffres météo
  -> voir ml_models/metrics_v2_calendar_lags.json pour ces nouveaux chiffres


## Pour le mémoire

- Deux pistes testées sans dépendance externe (pas d'API tierce, pas de clé,
  toujours disponibles) : plus de lags par station, et features calendaires
  (weekend, jours fériés).
- Le diagnostic en section 1 donne une lecture honnête de la quantité de
  données réellement disponible au moment du test — important pour
  interpréter correctement un résultat négatif (est-ce que la feature ne
  sert à rien, ou est-ce qu'on n'a simplement pas encore assez de données
  pour qu'elle serve ?).
- Même démarche de comparaison honnête à la baseline que pour la météo
  (notebook 04) : on ne garde que ce qui bat réellement la persistance,
  horizon par horizon.
- `ml_models/metrics_v2_calendar_lags.json` est un fichier séparé de
  `ml_models/metrics.json` pour comparer explicitement les deux approches
  sans écraser les résultats précédents — utile pour montrer la démarche
  itérative dans le mémoire.
